# PN3 standalone ARA parent/child prime-survival test

## Answer first

The frozen standalone ARA model did **not** beat its registered controls on the untouched `[1,000,000,000,1,010,000,000)` target. The parent forecast missed the aggregate candidate rate by `2.370%` and the pair rate by `1.535%`. ARA child states beat matched raw child states, but both child models were worse than the constant ARA parent. PNT and conditional Hardy–Littlewood remained best. Independent validation passed `118/118` checks.


## Frozen design

The standalone script contains no analytic prime-density law. It predicts the next parent rate as $\widehat p_9=p_8^2/p_7$, then uses local ARA states to redistribute that total with exact mean conservation. The target packet is hashed before the separate comparison script adds PNT or Hardy–Littlewood references.


In [1]:
import json
from pathlib import Path
import pandas as pd

HERE = Path.cwd()
results = json.loads((HERE / 'PN3_STANDALONE_ARA_RESULTS.json').read_text(encoding='utf-8'))
scores = pd.read_csv(HERE / 'PN3_STANDALONE_ARA_MODEL_SCORES.csv')
bootstrap = pd.read_csv(HERE / 'PN3_STANDALONE_ARA_BOOTSTRAP.csv')
validation = json.loads((HERE / 'PN3_INDEPENDENT_VALIDATION.json').read_text(encoding='utf-8'))
print('Test:', results['test_id'])
print('Target:', results['target_interval'])
print(f"Candidate events: {results['candidate_events']:,}; actual rate: {results['candidate_actual_rate']:.9f}")
print(f"Edge events: {results['edge_events']:,}; actual rate: {results['edge_actual_rate']:.9f}")
print('Criteria:', results['criteria'])


Test: PN3/STANDALONE-ARA-PARENT-CHILD/v1
Target: [1000000000, 1010000000]
Candidate events: 1,579,467; actual rate: 0.305450510
Edge events: 1,579,466; actual rate: 0.092529374
Criteria: {'candidate_P1_parent_recovery': False, 'edge_P1_parent_recovery': False, 'candidate_P2_child_redistribution': False, 'edge_P2_child_redistribution': False, 'candidate_P3_full_standalone': False, 'edge_P3_full_standalone': False}


## Parent recovery

The parent ARA recurrence was materially better than Home, but it did not reach the frozen 1% absolute-rate accuracy threshold.


In [2]:
rows = []
for task in ('candidate', 'edge'):
    item = results['parent_recovery'][task]
    rows.append({
        'task': task, 'actual_rate': item['actual_rate'], 'ara_prediction': item['ara'],
        'relative_error': item['ara_relative_rate_error'], 'ara_log_loss': item['ara_log_loss_bits'],
        'home_log_loss': item['home_log_loss_bits'], 'raw_additive_log_loss': item['raw_additive_log_loss_bits'],
    })
parent = pd.DataFrame(rows)
print(parent.to_string(index=False, float_format=lambda x: f'{x:.6f}'))
assert not results['criteria']['candidate_P1_parent_recovery']
assert not results['criteria']['edge_P1_parent_recovery']


     task  actual_rate  ara_prediction  relative_error  ara_log_loss  home_log_loss  raw_additive_log_loss
candidate     0.305451        0.298211        0.023702      0.888032       0.892499               0.888533
     edge     0.092529        0.091109        0.015346      0.444874       0.449380               0.445575


![Parent-rung recovery](PN3_STANDALONE_ARA_PARENT_RECOVERY.png)


## Primary child and established-reference comparisons

Positive gain favours ARA. The ARA child encodings beat raw encodings, but lose to parent-only and to the established analytic references.


In [3]:
view = bootstrap[['task', 'comparator', 'observed_gain_bits', 'ci95_low_bits', 'ci95_high_bits']]
print(view.to_string(index=False, float_format=lambda x: f'{x:.9f}'))
assert (bootstrap.loc[bootstrap.comparator.str.contains('raw_'), 'ci95_low_bits'] > 0).all()
assert (bootstrap.loc[bootstrap.comparator.isin(['ara_parent_only','pnt29_reference','hl29_reference']), 'ci95_high_bits'] < 0).all()


     task                                comparator  observed_gain_bits  ci95_low_bits  ci95_high_bits
candidate                   ara_parent_only           -0.000073689      -0.000095842        -0.000050295
candidate ara_parent_raw_stencil_child            0.000090078       0.000062844         0.000116327
candidate                    pnt29_reference           -0.000253481      -0.000289979        -0.000217519
     edge                   ara_parent_only           -0.000013902      -0.000022171        -0.000005301
     edge    ara_parent_raw_edge_child            0.000039072       0.000021894         0.000056062
     edge                     hl29_reference           -0.000031267      -0.000045892        -0.000016122


![Standalone model comparison](PN3_STANDALONE_ARA_MODEL_COMPARISON.png)

![Target-block calibration](PN3_STANDALONE_ARA_BLOCK_CALIBRATION.png)


## Independent validation


In [4]:
print(f"Independent checks: {validation['checks_passed']}/{validation['checks_total']}")
print('Packet hash:', validation['packet_sha256'])
assert validation['all_checks_passed']
assert validation['validator_imported_primary_scripts'] is False
assert results['packet_sha256_before'] == results['packet_sha256_after'] == validation['packet_sha256']


Independent checks: 118/118
Packet hash: 129832B150360C005DFF676A8F0140145BEB1E9DFCBF074BB4FC44ABDDDE1C6A


## Interpretation

PN3 falsifies this particular standalone recurrence-plus-child implementation. It does not erase the exact prime-wheel crosswalks from PN1–PN1I. The clean next question is parent-scale: develop an ARA coordinate for the slow density envelope on opened rungs, compare it with ordinary logarithmic/convergence models, and freeze it before another fresh interval. Do not tune the child geometry on this target.
